In [22]:
import requests
import tempfile
import PyPDF2
import re

def is_artifact(line: str) -> bool:
    punctuation = ".,:;?!-()[]{}'/\"$%"
    tokens = [t for t in line.split() if t]
    is_single_token_no_punctuation = len(tokens) == 1 and not any(p in tokens[0] for p in punctuation)
    if is_single_token_no_punctuation:
        return True

    artifacts = [
        r"Fmt \d+",  # Format markers
        r"Frm \d+",  # Frame markers
        r"Sfmt \d+",  # Section format markers
        r"VerDate.*",  # Version date lines
        r"HR \d+ .*",  # House resolution marker
    ]
    return any(re.search(pattern, line) for pattern in artifacts)

def clean_text(text: str) -> str:
    """Clean extracted text by removing artifacts and normalizing whitespace"""
    lines = text.split("\n")
    cleaned_lines = [
        line.strip() for line in lines if line.strip() and not is_artifact(line)
    ]

    # Remove line numbers at the start of any remaining lines
    cleaned_lines = [re.sub(r"^\d+ ", "", line) for line in cleaned_lines]

    # Rejoin with single newlines
    return "\n".join(cleaned_lines)


# Download and process in a single context manager
with tempfile.NamedTemporaryFile(delete=False) as pdf_file:
    # Download the PDF
    url = "https://www.congress.gov/118/bills/s2685/BILLS-118s2685enr.pdf"
    CHUNK_SIZE = 32768

    with requests.Session() as session:
        response = session.get(url, stream=True, timeout=30)
        response.raise_for_status()
        for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
            if chunk:
                pdf_file.write(chunk)
    pdf_file.flush()

    # Store the name for later use
    temp_filename = pdf_file.name

# Process the PDF
text_parts = []
try:
    with open(temp_filename, "rb") as file:
        pdf = PyPDF2.PdfReader(file)

        batch_size = 5
        for i in range(0, len(pdf.pages), batch_size):
            batch = pdf.pages[i : i + batch_size]
            for page in batch:
                try:
                    page_text = page.extract_text()
                    cleaned_text = clean_text(page_text)
                    if cleaned_text:
                        text_parts.append(cleaned_text)
                except Exception as e:
                    print(f"Error on page {i}: {e}")
                    continue

            import gc
            gc.collect()

    print(text_parts)

finally:
    # Clean up the temporary file
    import os
    try:
        os.unlink(temp_filename)
    except:
        pass

['S. 2685\nOne Hundred Eighteenth Congress\nof the\nUnited States of America\nAT THE SECOND SESSION\nBegun and held at the City of Washington on Wednesday,\nthe third day of January, two thousand and twenty four\nAn Act\nTo make data and internal guidance on excess personal property publicly available,\nand for other purposes.\nBe it enacted by the Senate and House of Representatives of\nthe United States of America in Congress assembled,\nSECTION 1. SHORT TITLE.\nThis Act may be cited as the ‘‘Reuse Excess Property Act’’.\nSEC. 2. REPORTING ON EXCESS PERSONAL PROPERTY.\n(a) I NGENERAL .—Subchapter II of chapter 5 of title 40, United\nStates Code, is amended—\n(1) in section 529—\n(A) in subsection (a), in the matter preceding paragraph\n(1), by inserting ‘‘and the Committee on Homeland Security and Governmental Affairs of the Senate and the Committee on Oversight and Accountability of the House of Represent-atives’’ after ‘‘Administrator of General Services’’; and\n(B) by adding at th